# Khảo sát nhanh output của Stage 2 (Pseudo-masks)
Chạy notebook này sau khi đã chạy lệnh `python stage2_cam/generate_pseudomasks.py`.

In [ ]:
import yaml
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import random

with open('configs/stage2_cam.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

# Khôi phục đường dẫn từ config
dataset_root = Path(cfg['dataset']['root'])
target_domain = cfg['dataset']['target_domain']
img_dir = dataset_root / target_domain / 'images'

mask_dir = Path(cfg['output']['pseudomask_dir'])
conf_dir = mask_dir / 'confidence'
cam_dir = Path(cfg['output']['cam_dir'])

all_masks = list(mask_dir.glob('*.png'))
print(f"Tìm thấy {len(all_masks)} pseudo-masks tại {mask_dir}")

In [ ]:
# Chọn ngẫu nhiên 5 ảnh để hiển thị
num_samples = min(5, len(all_masks))
if num_samples > 0:
    sample_paths = random.sample(all_masks, num_samples)
else:
    sample_paths = []

for mask_path in sample_paths:
    sample_id = mask_path.stem
    
    # Tìm ảnh gốc
    img_path_candidates = list(img_dir.glob(f"{sample_id}*"))
    if not img_path_candidates:
        print(f"Không tìm thấy ảnh gốc cho {sample_id}")
        continue
    
    img = Image.open(img_path_candidates[0]).convert('RGB')
    mask = np.array(Image.open(mask_path))
    
    conf_arr = np.zeros_like(mask, dtype=np.float32)
    conf_file = conf_dir / f"{sample_id}.npy"
    if conf_file.exists():
        conf_arr = np.load(conf_file)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(img)
    axes[0].set_title(f"Original: {sample_id}")
    axes[0].axis('off')
    
    # Mask map (255 = ignore/background)
    cmap = plt.cm.get_cmap('tab10').copy()
    cmap.set_over('white')
    cmap.set_under('black')
    
    im1 = axes[1].imshow(mask, cmap=cmap, vmin=0, vmax=3)
    axes[1].set_title("Pseudo Mask (Trắng = Ignore)")
    axes[1].axis('off')
    
    # Confidence map
    im2 = axes[2].imshow(conf_arr, cmap='jet', vmin=0, vmax=1)
    axes[2].set_title("Confidence Map")
    axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.show()